HealthRisk Analytics у PyTorch
Бізнес-контекст: компанія HealthRisk Analytics будує прототип моделі, яка за простими показниками клієнта прогнозує умовний рівень страхового ризику.

Завдання
У notebook є 3 основні місця для самостійного коду:

Створити тензори клієнтів і порахувати умовний ризик.
Навчити персептрон вручну протягом 20 епох.
Навчити той самий персептрон через SGD, заповнити таблицю loss і побудувати графік.

In [1]:
from pathlib import Path

import torch
from torch import nn
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
N_SAMPLES = 100
N_EPOCHS = 20
SGD_LR = 0.01
MANUAL_LR = 0.1
PLOT_PATH = Path("loss_healthrisk.png")

torch.manual_seed(RANDOM_STATE)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.12.1+cpu


Завдання 1. Базові операції з тензорами
У клітинці нижче:

створіть 1D-тензори age, weight_kg, height_m для 5 клієнтів. Значення можна взяти довільні, але реалістичні: наприклад вік у роках, вагу в кілограмах, зріст у метрах;
зберіть 2D-тензор client_features, де колонки відповідають віку, вазі та зросту;
створіть будь-який 3D-тензор client_batches з torch.randn(...);
порахуйте bmi = weight_kg / height_m ** 2;
порахуйте risk_score за формулою 0.02 * age + 0.03 * bmi + 0.001 * weight_kg. Тут кожна операція виконується одразу для всіх клієнтів;
створіть risk_weight = torch.tensor(0.03, requires_grad=True), потім порахуйте risk_with_grad = (0.02 * age + risk_weight * bmi + 0.001 * weight_kg).mean() і виведіть risk_with_grad.grad_fn.

In [ ]:
# TODO:
# 1. Створіть age, weight_kg, height_m з dtype=torch.float32
# 2. Зберіть client_features через torch.stack(..., dim=1)
# 3. Створіть client_batches через torch.randn(...)
# 4. Порахуйте bmi = weight_kg / height_m ** 2
# 5. Порахуйте risk_score = 0.02 * age + 0.03 * bmi + 0.001 * weight_kg
# 6. Створіть risk_weight = torch.tensor(0.03, requires_grad=True)
# 7. Порахуйте risk_with_grad = (0.02 * age + risk_weight * bmi + 0.001 * weight_kg).mean()
# 8. Виведіть risk_with_grad.grad_fn

# age = ...
# weight_kg = ...
# height_m = ...
# client_features = ...
# client_batches = ...
# bmi = ...
# risk_score = ...
# risk_weight = ...
# risk_with_grad = ...

Очікуваний результат

age.shape має бути torch.Size([5]).
client_features.shape має бути torch.Size([5, 3]).
client_batches має бути 3D-тензором.
risk_with_grad.grad_fn має показати, що PyTorch відстежує граф обчислень.
Перевірка таблиці клієнтів

In [ ]:
pd.DataFrame({
    "age": age.tolist(),
    "weight_kg": weight_kg.tolist(),
    "height_m": height_m.tolist(),
    "bmi": bmi.round(decimals=2).tolist(),
    "risk_score": risk_score.round(decimals=3).tolist(),
})

Дані для персептрона
Тепер згенеруємо 100 зразків з однією ознакою X: умовний BMI. Цільова змінна y - лінійна комбінація BMI та шум.

In [2]:
torch.manual_seed(RANDOM_STATE)

X = torch.randn(N_SAMPLES, 1)
y = 0.42 * X + 0.15 + 0.05 * torch.randn(N_SAMPLES, 1)
loss_fn = nn.MSELoss()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("target mean:", round(y.mean().item(), 4))

X shape: torch.Size([100, 1])
y shape: torch.Size([100, 1])
target mean: 0.1769


Завдання 2. Персептрон з ручним оновленням параметрів
У клітинці нижче реалізуйте один лінійний нейрон nn.Linear(1, 1) і навчіть його вручну.

Порядок на кожній епосі:

зробіть прогноз y_pred = manual_model(X);
порахуйте loss = loss_fn(y_pred, y);
викличте loss.backward();
у блоці with torch.no_grad(): оновіть кожен параметр через param -= MANUAL_LR * param.grad;
очистіть градієнти через manual_model.zero_grad();
додайте в history_manual словник з epoch, loss, weight, bias.

In [7]:
torch.manual_seed(RANDOM_STATE)

manual_model = nn.Linear(1, 1)
history_manual = []

initial_weight = manual_model.weight.item()
initial_bias = manual_model.bias.item()
initial_loss = loss_fn(manual_model(X), y).item()

for epoch in range(1, N_EPOCHS + 1):

    y_pred = manual_model(X)

    loss = loss_fn(y_pred, y)

    loss.backward()

    with torch.no_grad():
        for param in manual_model.parameters():
            param -= MANUAL_LR * param.grad

    manual_model.zero_grad()

    history_manual.append({
        "epoch": epoch,
        "loss": loss.item(),
        "weight": manual_model.weight.item(),
        "bias": manual_model.bias.item()
    })

manual_metrics = pd.DataFrame(history_manual)

print(manual_metrics.tail())

    epoch      loss    weight      bias
15     16  0.002503  0.427026  0.168924
16     17  0.002298  0.425577  0.165419
17     18  0.002169  0.424450  0.162632
18     19  0.002088  0.423574  0.160417
19     20  0.002037  0.422894  0.158654


Очікуваний результат

manual_metrics має містити 20 рядків.
loss має помітно зменшитися від початкового значення.
Фінальні weight і bias мають наблизитися до правила, за яким створено y: приблизно 0.42 і 0.15.
Коротке порівняння початку і кінця

In [4]:
pd.DataFrame([
    {
        "stage": "initial",
        "loss": initial_loss,
        "weight": initial_weight,
        "bias": initial_bias,
    },
    {
        "stage": "manual_final",
        "loss": manual_metrics.iloc[-1]["loss"],
        "weight": manual_metrics.iloc[-1]["weight"],
        "bias": manual_metrics.iloc[-1]["bias"],
    },
]).round(4)

,stage,loss,weight,bias
0,initial,0.6042,0.7645,0.8300
1,manual_final,0.0020,0.4229,0.1587


Завдання 3. Персептрон з оптимізатором SGD
Тепер навчіть таку саму модель через torch.optim.SGD(lr=0.01).

У клітинці нижче:

створіть sgd_model = nn.Linear(1, 1);
створіть optimizer = torch.optim.SGD(sgd_model.parameters(), lr=SGD_LR);
виконайте 20 епох;
на кожній епосі використайте порядок optimizer.zero_grad(), loss.backward(), optimizer.step();
створіть таблицю sgd_metrics з колонками epoch і loss_sgd.

In [ ]:
torch.manual_seed(RANDOM_STATE)

losses_sgd = []

# TODO:
# 1. Створіть sgd_model і optimizer.
# 2. Навчіть модель 20 епох.
# 3. Додайте loss кожної епохи в losses_sgd.
# 4. Створіть sgd_metrics і виведіть таблицю.

# sgd_model = ...
# optimizer = ...
# for epoch in range(1, N_EPOCHS + 1):
#     ...
# sgd_metrics = ...
# sgd_metrics

Очікуваний результат

Таблиця sgd_metrics має містити 20 епох. Значення loss_sgd мають поступово знижуватися, хоча через малий lr=0.01 збіжність може бути повільнішою, ніж у ручному прикладі з MANUAL_LR=0.1.

Графік втрат
Ця клітинка будує графік за sgd_metrics і зберігає його у файл loss_healthrisk.png.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sgd_metrics["epoch"], sgd_metrics["loss_sgd"], marker="o", label="SGD loss")
ax.set_title("HealthRisk perceptron: loss by epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.legend()

plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("Saved to:", PLOT_PATH.resolve())